[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MahkameSalimi/ISPRS-Tutorial/blob/main/intro_quantum_and_lstm/pennylane_intro_tutorial.ipynb)

# Hands-on Introduction to Qubits, Bloch Spheres, Gates, and Measurements with PennyLane


You will learn:

1. what a qubit is,
2. how to visualize a qubit on the Bloch sphere,
3. how rotations change the qubit,
4. how gates and measurements work on a simulator,
5. how two-qubit and three-qubit circuits create correlations,
6. the circuit materials needed before studying QLSTM: encoding, trainable rotations, ansatz layers, entanglement, expectation values, and data re-uploading.

## 0. Setup

Run this first. If PennyLane is not installed, uncomment the install line.

In [ ]:
# If needed, uncomment and run this line:
# !pip install pennylane matplotlib ipywidgets

import pennylane as qml
from pennylane import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

np.set_printoptions(precision=3, suppress=True)

### Helper functions for Bloch sphere visualization

For a one-qubit pure state

$$
|\psi\rangle = \alpha |0\rangle + \beta |1\rangle,
$$

the Bloch vector is

$$
x = 2\mathrm{Re}(\alpha^*\beta), \qquad
y = 2\mathrm{Im}(\alpha^*\beta), \qquad
z = |\alpha|^2 - |\beta|^2.
$$

PennyLane gives us the statevector using `qml.state()`. We then convert it to a Bloch vector and plot it.

In [ ]:
def state_to_bloch_vector(state):
    # Return the Bloch vector [x, y, z] for a one-qubit pure state.
    state = np.asarray(state, dtype=complex)
    alpha, beta = state[0], state[1]
    x = 2 * np.real(np.conj(alpha) * beta)
    y = 2 * np.imag(np.conj(alpha) * beta)
    z = np.abs(alpha)**2 - np.abs(beta)**2
    return np.array([x, y, z], dtype=float)


def plot_bloch_vector(vector, title="Bloch sphere"):
    # Plot a simple Bloch sphere with one state vector.
    x, y, z = vector

    fig = plt.figure(figsize=(5, 5))
    ax = fig.add_subplot(111, projection="3d")

    u = np.linspace(0, 2 * np.pi, 80)
    v = np.linspace(0, np.pi, 40)
    xs = np.outer(np.cos(u), np.sin(v))
    ys = np.outer(np.sin(u), np.sin(v))
    zs = np.outer(np.ones_like(u), np.cos(v))
    ax.plot_surface(xs, ys, zs, alpha=0.08, linewidth=0)

    ax.quiver(0, 0, 0, 1.1, 0, 0, arrow_length_ratio=0.08)
    ax.quiver(0, 0, 0, 0, 1.1, 0, arrow_length_ratio=0.08)
    ax.quiver(0, 0, 0, 0, 0, 1.1, arrow_length_ratio=0.08)

    ax.text(1.2, 0, 0, "x")
    ax.text(0, 1.2, 0, "y")
    ax.text(0, 0, 1.2, "|0>")
    ax.text(0, 0, -1.2, "|1>")

    ax.quiver(0, 0, 0, x, y, z, linewidth=3, arrow_length_ratio=0.12, color= "red")

    ax.set_xlim([-1.2, 1.2])
    ax.set_ylim([-1.2, 1.2])
    ax.set_zlim([-1.2, 1.2])
    ax.set_box_aspect([1, 1, 1])
    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")
    plt.show()


def print_state_and_plot(state, title="Bloch sphere"):
    print("Statevector:", state)
    bloch_vector = state_to_bloch_vector(state)
    print("Bloch vector [x, y, z]:", bloch_vector)
    plot_bloch_vector(bloch_vector, title=title)

## 1. What is a qubit?

A classical bit is either `0` or `1`.

A qubit can be in a combination of two basis states:

$$
|\psi\rangle = \alpha |0\rangle + \beta |1\rangle.
$$

When we measure the qubit:

- probability of getting `0` is $|\alpha|^2$,
- probability of getting `1` is $|\beta|^2$.

The amplitudes must satisfy:

$$
|\alpha|^2 + |\beta|^2 = 1.
$$

### 1.1 The state $|0\rangle$

In PennyLane, a fresh qubit starts in state $|0\rangle$.

In [ ]:
dev = qml.device("default.qubit", wires=1)

@qml.qnode(dev)
def zero_state():
    return qml.state()

state = zero_state()
print_state_and_plot(state, title="State |0>")

On the Bloch sphere, $|0\rangle$ points upward along the positive $z$ axis.

### 1.2 The state $|1\rangle$

The `PauliX` gate flips $|0\rangle$ to $|1\rangle$.

In [ ]:
@qml.qnode(dev)
def one_state():
    qml.PauliX(wires=0)
    return qml.state()

state = one_state()
print_state_and_plot(state, title="State |1>")

On the Bloch sphere, $|1\rangle$ points downward along the negative $z$ axis.

## 2. Superposition with the Hadamard gate

The Hadamard gate creates

$$
|+\rangle = \frac{|0\rangle + |1\rangle}{\sqrt{2}}.
$$

This state has equal probability of being measured as `0` or `1`.

In [ ]:
@qml.qnode(dev)
def plus_state():
    qml.Hadamard(wires=0)
    return qml.state()

state = plus_state()
print_state_and_plot(state, title="State |+>")

The $|+\rangle$ state points along the positive $x$ axis.

## 3. Rotations on the Bloch sphere

One-qubit rotation gates are central in quantum computation:

$$
R_x(\theta), \qquad R_y(\theta), \qquad R_z(\theta).
$$

Each one rotates the qubit around one axis of the Bloch sphere.

### 3.1 Rotation around the $y$ axis

In [ ]:
def ry_state(theta):
    @qml.qnode(dev)
    def circuit():
        qml.RY(theta, wires=0)
        return qml.state()
    return circuit()

for theta in [0, np.pi/4, np.pi/2, np.pi]:
    state = ry_state(theta)
    print(f"theta = {theta:.3f}")
    print_state_and_plot(state, title=f"RY({theta:.2f}) |0>")

Important cases:

- $R_y(0)|0\rangle = |0\rangle$
- $R_y(\pi/2)|0\rangle$ gives an equal superposition
- $R_y(\pi)|0\rangle = |1\rangle$

### 3.2 Optional interactive Bloch sphere

Run this only if your environment supports widgets.

In [ ]:
try:
    from ipywidgets import interact, FloatSlider

    def show_ry(theta):
        state = ry_state(theta)
        print_state_and_plot(state, title=f"RY({theta:.2f}) |0>")

    interact(
        show_ry,
        theta=FloatSlider(min=0, max=2*np.pi, step=0.1, value=0, description="theta")
    )
except Exception as e:
    print("Interactive widgets are not available here.")
    print(e)

### 3.3 Compare $R_x$, $R_y$, and $R_z$

In [ ]:
def rotation_state(axis, theta):
    @qml.qnode(dev)
    def circuit():
        if axis == "x":
            qml.RX(theta, wires=0)
        elif axis == "y":
            qml.RY(theta, wires=0)
        elif axis == "z":
            qml.RZ(theta, wires=0)
        return qml.state()
    return circuit()

for axis in ["x", "y", "z"]:
    state = rotation_state(axis, np.pi/2)
    print(f"Rotation around {axis}-axis")
    print_state_and_plot(state, title=f"R{axis.upper()}(pi/2) |0>")

Notice that $R_z$ does not visibly move $|0\rangle$ on the Bloch sphere.

This is because $|0\rangle$ already lies on the $z$ axis. Rotating a point around the same axis does not change its visible position.

## 4. Common one-qubit gates

| Gate | PennyLane operation | Main effect |
|---|---|---|
| X | `qml.PauliX` | flips $\|0\rangle \leftrightarrow \|1\rangle$ |
| Y | `qml.PauliY` | flip plus phase change |
| Z | `qml.PauliZ` | phase flip |
| H | `qml.Hadamard` | creates superposition |
| S | `qml.S` | phase gate: $\pi/2$ phase |
| T | `qml.T` | phase gate: $\pi/4$ phase |
| RX | `qml.RX(theta)` | rotation around x-axis |
| RY | `qml.RY(theta)` | rotation around y-axis |
| RZ | `qml.RZ(theta)` | rotation around z-axis |

In [ ]:
def apply_gate_and_show(gate_name):
    @qml.qnode(dev)
    def circuit():
        if gate_name == "X":
            qml.PauliX(wires=0)
        elif gate_name == "Y":
            qml.PauliY(wires=0)
        elif gate_name == "Z":
            qml.PauliZ(wires=0)
        elif gate_name == "H":
            qml.Hadamard(wires=0)
        elif gate_name == "S":
            qml.S(wires=0)
        elif gate_name == "T":
            qml.T(wires=0)
        return qml.state()

    state = circuit()
    print_state_and_plot(state, title=f"{gate_name}|0>")

for gate in ["X", "Y", "Z", "H", "S", "T"]:
    apply_gate_and_show(gate)

Some phase gates do not visibly change $|0\rangle$, but they become important when applied to superposition states.

## 5. Measurement on a simulator

A statevector tells us the full quantum state.

But quantum computers give classical measurement outcomes.

In PennyLane:

- `qml.probs` gives exact probabilities,
- `qml.sample` gives finite-shot samples.

### 5.1 Helper function for running measurements

In [ ]:
def run_measurement(circuit_builder, wires=1, shots=1024):
    dev_probs = qml.device("default.qubit", wires=wires)
    dev_shots = qml.device("default.qubit", wires=wires, shots=shots)

    @qml.qnode(dev_probs)
    def prob_circuit():
        circuit_builder()
        return qml.probs(wires=range(wires))

    @qml.qnode(dev_shots)
    def sample_circuit():
        circuit_builder()
        return qml.sample(wires=range(wires))

    probs = prob_circuit()
    samples = sample_circuit()

    if wires == 1:
        bitstrings = [str(int(s)) for s in samples]
    else:
        bitstrings = ["".join(str(int(bit)) for bit in row) for row in samples]

    counts = Counter(bitstrings)
    return probs, counts


def plot_counts(counts, title="Measurement counts"):
    labels = sorted(counts.keys())
    values = [counts[label] for label in labels]
    plt.figure(figsize=(5, 3))
    plt.bar(labels, values)
    plt.xlabel("Outcome")
    plt.ylabel("Counts")
    plt.title(title)
    plt.show()

### 5.2 Measure $|0\rangle$

In [ ]:
def circuit_zero():
    pass

probs, counts = run_measurement(circuit_zero, wires=1, shots=1024)
print("Exact probabilities [P(0), P(1)] =", probs)
print("Counts =", counts)
plot_counts(counts, title="Measuring |0>")

### 5.3 Measure $|1\rangle$

In [ ]:
def circuit_one():
    qml.PauliX(wires=0)

probs, counts = run_measurement(circuit_one, wires=1, shots=1024)
print("Exact probabilities [P(0), P(1)] =", probs)
print("Counts =", counts)
plot_counts(counts, title="Measuring |1>")

### 5.4 Measure $|+\rangle$

In [ ]:
def circuit_plus():
    qml.Hadamard(wires=0)

probs, counts = run_measurement(circuit_plus, wires=1, shots=1024)
print("Exact probabilities [P(0), P(1)] =", probs)
print("Counts =", counts)
plot_counts(counts, title="Measuring |+>")

### 5.5 Measure after a rotation

For the state $R_y(\theta)|0\rangle$,

$$
P(1) = \sin^2(\theta/2).
$$

In [ ]:
theta = np.pi / 3

def circuit_ry():
    qml.RY(theta, wires=0)

probs, counts = run_measurement(circuit_ry, wires=1, shots=2048)
print("theta =", theta)
print("Exact probabilities [P(0), P(1)] =", probs)
print("Expected P(1) =", np.sin(theta/2)**2)
print("Counts =", counts)
plot_counts(counts, title="Measuring RY(theta)|0>")

## 6. Two-qubit circuits and entanglement

Now we create a Bell state:

$$
|\Phi^+\rangle = \frac{|00\rangle + |11\rangle}{\sqrt{2}}.
$$

This is an entangled state. The two qubits are correlated.

In [ ]:
dev2 = qml.device("default.qubit", wires=2)

@qml.qnode(dev2)
def bell_state():
    qml.Hadamard(wires=0)
    qml.CNOT(wires=[0, 1])
    return qml.state()

state = bell_state()
print("Bell statevector:")
print(state)

@qml.qnode(dev2)
def draw_bell():
    qml.Hadamard(wires=0)
    qml.CNOT(wires=[0, 1])
    return qml.probs(wires=[0, 1])

qml.draw_mpl(draw_bell)()
plt.show()

In [ ]:
def circuit_bell():
    qml.Hadamard(wires=0)
    qml.CNOT(wires=[0, 1])

probs, counts = run_measurement(circuit_bell, wires=2, shots=2048)
print("Exact probabilities [P(00), P(01), P(10), P(11)] =")
print(probs)
print("Counts =", counts)
plot_counts(counts, title="Bell-state measurement")

### Important note about Bloch spheres for entangled states

A single-qubit Bloch sphere is perfect for visualizing one qubit.

But an entangled two-qubit state cannot be fully visualized as two independent Bloch vectors.

## 7. A more complicated circuit: three-qubit GHZ state

The GHZ state is

$$
|\mathrm{GHZ}\rangle = \frac{|000\rangle + |111\rangle}{\sqrt{2}}.
$$

In [ ]:
dev3 = qml.device("default.qubit", wires=3)

@qml.qnode(dev3)
def ghz_state():
    qml.Hadamard(wires=0)
    qml.CNOT(wires=[0, 1])
    qml.CNOT(wires=[1, 2])
    return qml.state()

state = ghz_state()
print("GHZ statevector:")
print(state)

@qml.qnode(dev3)
def draw_ghz():
    qml.Hadamard(wires=0)
    qml.CNOT(wires=[0, 1])
    qml.CNOT(wires=[1, 2])
    return qml.probs(wires=[0, 1, 2])

qml.draw_mpl(draw_ghz)()
plt.show()

In [ ]:
def circuit_ghz():
    qml.Hadamard(wires=0)
    qml.CNOT(wires=[0, 1])
    qml.CNOT(wires=[1, 2])

probs, counts = run_measurement(circuit_ghz, wires=3, shots=4096)
print("Exact probabilities:")
print(probs)
print("Counts =", counts)
plot_counts(counts, title="GHZ-state measurement")

## 8. Exercises

Try these before looking at the solutions.

### Exercise 1: Create $|1\rangle$

Create a PennyLane circuit that starts from $|0\rangle$, applies one gate, and returns the state $|1\rangle$.

In [ ]:
# Your code here

### Exercise 2: Create an equal superposition

Create $ |+\rangle=(|0\rangle+|1\rangle)/\sqrt{2} $. Then measure it 1000 times. What counts do you expect?

In [ ]:
# Your code here

### Exercise 3: Find a rotation angle

Use $R_y(\theta)$ to create a state where $P(1)\approx 0.25$. Hint: $P(1)=\sin^2(\theta/2)$.

In [ ]:
# Your code here

### Exercise 4: Bell state

Create $(|00\rangle+|11\rangle)/\sqrt{2}$. Measure it 2000 times. Which bitstrings should appear?

In [ ]:
# Your code here

### Exercise 5: Make your own three-qubit circuit

Create a three-qubit circuit with at least one Hadamard gate, one rotation gate, and one CNOT gate. Return the state and measure the circuit.

In [ ]:
# Your code here

## 9. Solutions

### Solution 1

In [ ]:
@qml.qnode(dev)
def solution_1():
    qml.PauliX(wires=0)
    return qml.state()

state = solution_1()
print_state_and_plot(state, title="Solution 1: |1>")

### Solution 2

In [ ]:
def solution_2_circuit():
    qml.Hadamard(wires=0)

probs, counts = run_measurement(solution_2_circuit, wires=1, shots=1000)
print("Exact probabilities:", probs)
print("Counts:", counts)
plot_counts(counts, title="Solution 2: measuring |+>")

# Expected result: about 50% zeros and 50% ones.

### Solution 3

In [ ]:
theta = np.pi / 3

def solution_3_circuit():
    qml.RY(theta, wires=0)

probs, counts = run_measurement(solution_3_circuit, wires=1, shots=2000)
print("theta = pi/3")
print("Exact probabilities:", probs)
print("Counts:", counts)
plot_counts(counts, title="Solution 3: P(1) ≈ 0.25")

### Solution 4

In [ ]:
def solution_4_circuit():
    qml.Hadamard(wires=0)
    qml.CNOT(wires=[0, 1])

probs, counts = run_measurement(solution_4_circuit, wires=2, shots=2000)
print("Exact probabilities:", probs)
print("Counts:", counts)
plot_counts(counts, title="Solution 4: Bell state")

# Expected result: mostly 00 and 11.

### Solution 5: one possible answer

In [ ]:
@qml.qnode(dev3)
def solution_5_state():
    qml.Hadamard(wires=0)
    qml.RY(np.pi/4, wires=1)
    qml.CNOT(wires=[0, 1])
    qml.CNOT(wires=[1, 2])
    return qml.state()

state = solution_5_state()
print("Statevector:")
print(state)

@qml.qnode(dev3)
def solution_5_draw():
    qml.Hadamard(wires=0)
    qml.RY(np.pi/4, wires=1)
    qml.CNOT(wires=[0, 1])
    qml.CNOT(wires=[1, 2])
    return qml.probs(wires=[0, 1, 2])

qml.draw_mpl(solution_5_draw)()
plt.show()

def solution_5_circuit():
    qml.Hadamard(wires=0)
    qml.RY(np.pi/4, wires=1)
    qml.CNOT(wires=[0, 1])
    qml.CNOT(wires=[1, 2])

probs, counts = run_measurement(solution_5_circuit, wires=3, shots=3000)
print("Exact probabilities:", probs)
print("Counts:", counts)
plot_counts(counts, title="Solution 5: custom 3-qubit circuit")

## 10. Circuit materials you need before QLSTM

Before studying a QLSTM implementation, learners should understand the circuit blocks that often appear inside a QLSTM quantum layer.

A QLSTM may use a **variational quantum circuit** as a trainable component.

The basic pipeline is:

$$
\text{classical data} \rightarrow \text{encoding gates} \rightarrow \text{trainable quantum circuit} \rightarrow \text{expectation values} \rightarrow \text{classical output}.
$$

This section teaches those circuit materials only.

### 10.1 From classical data to angles

A simple method is **angle encoding**:

$$
x_i \mapsto R_y(x_i).
$$

So each classical feature becomes a rotation angle.

In [ ]:
x = np.array([0.2, 0.7, -0.4])
dev_enc = qml.device("default.qubit", wires=3)

@qml.qnode(dev_enc)
def angle_encoding_circuit(x):
    for i in range(3):
        qml.RY(x[i], wires=i)
    return qml.state()

state = angle_encoding_circuit(x)
print("Encoded state:")
print(state)

qml.draw_mpl(angle_encoding_circuit)(x)
plt.show()

In [ ]:
feature_value = 0.7

@qml.qnode(dev)
def one_feature_encoding(x):
    qml.RY(x, wires=0)
    return qml.state()

state = one_feature_encoding(feature_value)
print_state_and_plot(state, title="One feature encoded as RY(x)")

### 10.2 Add trainable rotations

A variational quantum circuit has trainable parameters:

$$
R_y(\theta_i), \qquad R_z(\phi_i).
$$

In [ ]:
params = np.array([
    [0.1, 0.2],
    [0.3, 0.4],
    [0.5, 0.6]
])

@qml.qnode(dev_enc)
def trainable_rotation_circuit(x, params):
    for i in range(3):
        qml.RY(x[i], wires=i)

    for i in range(3):
        qml.RY(params[i, 0], wires=i)
        qml.RZ(params[i, 1], wires=i)

    return qml.state()

state = trainable_rotation_circuit(x, params)
print("State after encoding + trainable rotations:")
print(state)

qml.draw_mpl(trainable_rotation_circuit)(x, params)
plt.show()

### 10.3 Entangling layer

Entangling gates, such as `CNOT`, allow qubits to become correlated.

In [ ]:
@qml.qnode(dev_enc)
def entangling_circuit(x, params):
    for i in range(3):
        qml.RY(x[i], wires=i)

    for i in range(3):
        qml.RY(params[i, 0], wires=i)
        qml.RZ(params[i, 1], wires=i)

    qml.CNOT(wires=[0, 1])
    qml.CNOT(wires=[1, 2])

    return qml.state()

state = entangling_circuit(x, params)
print("State after encoding + trainable rotations + entanglement:")
print(state)

qml.draw_mpl(entangling_circuit)(x, params)
plt.show()

### 10.4 Repeated variational layers: a simple ansatz

An **ansatz** is the chosen structure of a parameterized circuit.

A common pattern is:

1. encode data,
2. apply trainable rotations,
3. apply entangling gates,
4. repeat steps 2 and 3 several times.

In [ ]:
def variational_layer(layer_params):
    n_qubits = layer_params.shape[0]

    for i in range(n_qubits):
        qml.RY(layer_params[i, 0], wires=i)
        qml.RZ(layer_params[i, 1], wires=i)

    for i in range(n_qubits - 1):
        qml.CNOT(wires=[i, i + 1])


n_qubits = 3
n_layers = 2
params_layers = np.array([
    [[0.1, 0.2], [0.3, 0.4], [0.5, 0.6]],
    [[0.7, 0.8], [0.9, 1.0], [1.1, 1.2]],
])

@qml.qnode(dev_enc)
def ansatz_circuit(x, params_layers):
    for i in range(n_qubits):
        qml.RY(x[i], wires=i)

    for l in range(n_layers):
        variational_layer(params_layers[l])

    return qml.state()

state = ansatz_circuit(x, params_layers)
print("State after a 2-layer ansatz:")
print(state)

qml.draw_mpl(ansatz_circuit)(x, params_layers)
plt.show()

### 10.5 From quantum state to classical output

Quantum machine learning often uses expectation values such as

$$
\langle Z_0 \rangle, \qquad \langle Z_1 \rangle, \qquad \langle Z_2 \rangle.
$$

Each value is between $-1$ and $+1$.

In [ ]:
@qml.qnode(dev_enc)
def quantum_layer_outputs(x, params_layers):
    for i in range(n_qubits):
        qml.RY(x[i], wires=i)

    for l in range(n_layers):
        variational_layer(params_layers[l])

    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

outputs = quantum_layer_outputs(x, params_layers)
print("Quantum layer outputs:", outputs)

### 10.6 A tiny quantum layer function

This is not a QLSTM. It is only the quantum-layer building block.

In [ ]:
def quantum_layer(x, params_layers):
    return np.array(quantum_layer_outputs(x, params_layers))

x_example = np.array([0.1, 0.5, -0.2])
out = quantum_layer(x_example, params_layers)
print("Input:", x_example)
print("Quantum layer output:", out)

### 10.7 Data re-uploading

Sometimes the same classical input is encoded more than once.

In [ ]:
@qml.qnode(dev_enc)
def data_reuploading_circuit(x, params_layers):
    for l in range(n_layers):
        for i in range(n_qubits):
            qml.RY(x[i], wires=i)

        variational_layer(params_layers[l])

    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

outputs = data_reuploading_circuit(x, params_layers)
print("Outputs with data re-uploading:", outputs)

qml.draw_mpl(data_reuploading_circuit)(x, params_layers)
plt.show()

### 10.8 Why this prepares learners for QLSTM

A classical LSTM uses trainable transformations to compute:

- forget gate,
- input gate,
- candidate memory,
- output gate.

A QLSTM may replace some of those trainable transformations with quantum variational circuits.

The key bridge is:

$$
\text{classical vector} \rightarrow \text{quantum circuit} \rightarrow \text{expectation values} \rightarrow \text{classical vector}.
$$

### 10.9 Exercises: circuit materials for QLSTM

#### Exercise 6: Encoding

Create a 4-qubit circuit that encodes

$$
x = [0.1, 0.2, 0.3, 0.4]
$$

using $R_y$ gates.

#### Exercise 7: Add trainable parameters

Add one trainable $R_z(\theta_i)$ gate after each encoding gate.

#### Exercise 8: Add entanglement

Add a chain of CNOT gates:

$$
0 \rightarrow 1, \qquad 1 \rightarrow 2, \qquad 2 \rightarrow 3.
$$

#### Exercise 9: Return expectation values

Return $\langle Z_i \rangle$ for each qubit.

#### Exercise 10: Data re-uploading

Repeat encoding plus trainable rotations twice.

### 10.10 Suggested answers

In [ ]:
x4 = np.array([0.1, 0.2, 0.3, 0.4])
dev4 = qml.device("default.qubit", wires=4)

@qml.qnode(dev4)
def ex6_encoding(x):
    for i in range(4):
        qml.RY(x[i], wires=i)
    return qml.state()

print("Exercise 6 state:")
print(ex6_encoding(x4))
qml.draw_mpl(ex6_encoding)(x4)
plt.show()

In [ ]:
theta4 = np.array([0.5, 0.6, 0.7, 0.8])

@qml.qnode(dev4)
def ex7_trainable(x, theta):
    for i in range(4):
        qml.RY(x[i], wires=i)
        qml.RZ(theta[i], wires=i)
    return qml.state()

print("Exercise 7 state:")
print(ex7_trainable(x4, theta4))
qml.draw_mpl(ex7_trainable)(x4, theta4)
plt.show()

In [ ]:
@qml.qnode(dev4)
def ex8_entangled(x, theta):
    for i in range(4):
        qml.RY(x[i], wires=i)
        qml.RZ(theta[i], wires=i)

    qml.CNOT(wires=[0, 1])
    qml.CNOT(wires=[1, 2])
    qml.CNOT(wires=[2, 3])

    return qml.state()

print("Exercise 8 state:")
print(ex8_entangled(x4, theta4))
qml.draw_mpl(ex8_entangled)(x4, theta4)
plt.show()

In [ ]:
@qml.qnode(dev4)
def ex9_expectations(x, theta):
    for i in range(4):
        qml.RY(x[i], wires=i)
        qml.RZ(theta[i], wires=i)

    qml.CNOT(wires=[0, 1])
    qml.CNOT(wires=[1, 2])
    qml.CNOT(wires=[2, 3])

    return [qml.expval(qml.PauliZ(i)) for i in range(4)]

print("Exercise 9 expectation values:")
print(ex9_expectations(x4, theta4))

In [ ]:
params_2_layers = np.array([
    [0.5, 0.6, 0.7, 0.8],
    [0.9, 1.0, 1.1, 1.2],
])

@qml.qnode(dev4)
def ex10_reuploading(x, params_2_layers):
    for layer in range(2):
        for i in range(4):
            qml.RY(x[i], wires=i)

        for i in range(4):
            qml.RZ(params_2_layers[layer, i], wires=i)

        qml.CNOT(wires=[0, 1])
        qml.CNOT(wires=[1, 2])
        qml.CNOT(wires=[2, 3])

    return [qml.expval(qml.PauliZ(i)) for i in range(4)]

print("Exercise 10 outputs:")
print(ex10_reuploading(x4, params_2_layers))
qml.draw_mpl(ex10_reuploading)(x4, params_2_layers)
plt.show()

### 10.11 Why this matters for QLSTM

A classical LSTM has gates such as:

- forget gate,
- input gate,
- candidate/update gate,
- output gate.

Each gate is normally computed using classical neural-network layers.

In a QLSTM, the main idea is to replace some of those classical layers with **variational quantum circuits**.

So instead of only doing something like:

$$
f_t = \sigma(W_f [h_{t-1}, x_t] + b_f),
$$

we may use a quantum circuit as part of the gate computation:

$$
f_t = \sigma(\text{ClassicalPostProcess}(\text{VQC}_f([h_{t-1}, x_t]))).
$$

The VQC acts like a trainable feature map.

## 11. questions

1. Why does measurement give different outcomes over many shots?
2. Why do we need many shots to estimate probabilities?
3. Why is a Bloch sphere not enough to fully describe entangled states?
4. Why are parameterized rotations useful for machine learning?
5. What is the difference between measuring bitstrings and returning expectation values?
6. Why might data re-uploading make a variational circuit more expressive?
7. In a QLSTM, which part is quantum and which part is classical?